# EXP-030: TabNet Optuna 하이퍼파라미터 탐색 + 4모델 앙상블

- **예상 소요시간**: 4–6시간 (timeout=3.5h Optuna + GBDT ~30–60min)
- **목적**: TabNet 기본 파라미터(EXP-029) 대신 Optuna로 구조 자체를 최적화
- **탐색 대상**: n_d/n_a, n_steps, gamma, lr, lambda_sparse, mask_type, batch_size
- **피처**: v1 (85개) — EXP-020 동일 기준선
- **기준선**: EXP-020 OOF AUC = 0.74068, 제출 = 0.74217
- **중단/재실행 안전**: Optuna study가 SQLite에 저장됨 → 재실행 시 이어서 탐색

In [1]:
import os
# libomp(PyTorch) + libgomp(numpy/scipy) 중복 로드 충돌 방지 → 모든 import 전에 설정
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS']       = '1'
os.environ['MKL_NUM_THREADS']       = '1'

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
import time

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

import torch
torch.set_num_threads(1)  # 런타임 레벨에서 OpenMP 스레드 1개로 고정
from pytorch_tabnet.tab_model import TabNetClassifier

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR  = Path('../data/raw')
OUT_DIR   = Path('../data/submissions')
DOCS_DIR  = Path('../docs')
CKPT_DIR  = Path('../data/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TARGET  = '임신 성공 여부'
SEED    = 42
N_FOLDS = 5
EXP_NO  = 30
AUTHOR  = '조여진'
CV_STR  = f'Stratified {N_FOLDS}-Fold'

# Optuna 탐색 설정
OPTUNA_TIMEOUT = int(3.5 * 3600)  # 3.5시간 후 새 trial 중단 (진행 중인 trial은 완료)
OPTUNA_N_TRIALS = 50              # 상한선 (timeout이 먼저 걸릴 가능성 높음)
OPTUNA_DB = str(CKPT_DIR / 'tabnet_optuna_exp030.db')

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')
print(f'torch threads: {torch.get_num_threads()}')

train: (256351, 69)  /  test: (90067, 68)
torch threads: 1


## 1. 피처 준비 (v1, EXP-020 동일)

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

X_arr  = X_train.values.astype(np.float32)
y_arr  = y_train.values
Xt_arr = X_test.values.astype(np.float32)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')
print(f'클래스 불균형 비율 (neg:pos) = {neg_pos_ratio:.2f}')

X_train: (256351, 85)  /  X_test: (90067, 85)
클래스 불균형 비율 (neg:pos) = 2.87


## 2. TabNet Optuna 하이퍼파라미터 탐색

- `timeout=3.5h`: 3.5시간 후 새 trial 생성 중단 (진행 중 trial은 끝까지 완료)
- `n_trials=50`: 상한선
- SQLite 저장: 중단 후 재실행 시 `load_if_exists=True`로 이어서 탐색
- 탐색 공간: n_d, n_a, n_steps, gamma, lr, weight_decay, lambda_sparse, mask_type, batch_size

In [3]:
def tabnet_cv_auc(params, batch_size, n_folds=N_FOLDS):
    """주어진 파라미터로 StratifiedKFold OOF AUC 반환"""
    oof = np.zeros(len(X_arr))
    fold_skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    for tr_idx, val_idx in fold_skf.split(X_arr, y_arr):
        X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]
        imputer = SimpleImputer(strategy='median')
        X_tr_i  = imputer.fit_transform(X_tr)
        X_val_i = imputer.transform(X_val)
        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_tr_i)
        X_val_s = scaler.transform(X_val_i)
        model = TabNetClassifier(**params)
        model.fit(
            X_tr_s, y_tr,
            eval_set=[(X_val_s, y_val)],
            eval_metric=['auc'],
            max_epochs=200,
            patience=20,
            batch_size=batch_size,
            virtual_batch_size=max(64, batch_size // 8),
            weights=1,
            num_workers=0,  # macOS 멀티프로세싱 크래시 방지
        )
        oof[val_idx] = model.predict_proba(X_val_s)[:, 1]
    return roc_auc_score(y_arr, oof)


def tabnet_objective(trial):
    n_d = trial.suggest_categorical('n_d', [16, 32, 64, 128])
    n_a = trial.suggest_categorical('n_a', [16, 32, 64, 128])
    params = dict(
        n_d=n_d,
        n_a=n_a,
        n_steps=trial.suggest_int('n_steps', 2, 6),
        gamma=trial.suggest_float('gamma', 1.0, 2.0),
        n_independent=trial.suggest_int('n_independent', 1, 4),
        n_shared=trial.suggest_int('n_shared', 1, 4),
        lambda_sparse=trial.suggest_float('lambda_sparse', 1e-5, 1e-2, log=True),
        optimizer_params=dict(
            lr=trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        ),
        mask_type=trial.suggest_categorical('mask_type', ['sparsemax', 'entmax']),
        seed=SEED,
        verbose=0,
    )
    batch_size = trial.suggest_categorical('batch_size', [2048, 4096, 8192])
    auc = tabnet_cv_auc(params, batch_size)
    print(f'  Trial {trial.number:>2d}  AUC={auc:.5f}  '
          f'n_d={n_d} n_a={n_a} steps={params["n_steps"]} '
          f'gamma={params["gamma"]:.2f} lr={params["optimizer_params"]["lr"]:.4f} '
          f'bs={batch_size}')
    return auc


storage = f'sqlite:///{OPTUNA_DB}'
study = optuna.create_study(
    direction='maximize',
    study_name='tabnet_exp030',
    storage=storage,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

already_done = len(study.trials)
print(f'탐색 시작 (기존 완료 trials: {already_done})')
print(f'timeout={OPTUNA_TIMEOUT//3600:.1f}h, n_trials 상한={OPTUNA_N_TRIALS}')
t0 = time.time()

study.optimize(
    tabnet_objective,
    n_trials=OPTUNA_N_TRIALS,
    timeout=OPTUNA_TIMEOUT,
    show_progress_bar=False,
)

elapsed = (time.time() - t0) / 60
print(f'\n탐색 완료: 총 {len(study.trials)}회 trial ({elapsed:.1f}분 소요)')
print(f'Best AUC : {study.best_value:.5f}')
print(f'Best params: {study.best_params}')

탐색 시작 (기존 완료 trials: 3)
timeout=3.0h, n_trials 상한=50

Early stopping occurred at epoch 114 with best_epoch = 94 and best_val_0_auc = 0.73166
Stop training because you reached max_epochs = 200 with best_epoch = 194 and best_val_0_auc = 0.73675
Stop training because you reached max_epochs = 200 with best_epoch = 180 and best_val_0_auc = 0.73357

Early stopping occurred at epoch 191 with best_epoch = 171 and best_val_0_auc = 0.7325
Stop training because you reached max_epochs = 200 with best_epoch = 193 and best_val_0_auc = 0.73485
  Trial  3  AUC=0.73381  n_d=32 n_a=128 steps=5 gamma=1.71 lr=0.0002 bs=2048

탐색 완료: 총 4회 trial (479.3분 소요)
Best AUC : 0.73381
Best params: {'n_d': 32, 'n_a': 128, 'n_steps': 5, 'gamma': 1.7080725777960455, 'n_independent': 1, 'n_shared': 4, 'lambda_sparse': 0.00314288089084011, 'lr': 0.00022948683681130568, 'weight_decay': 3.5113563139704077e-06, 'mask_type': 'entmax', 'batch_size': 2048}


## 3. 최적 TabNet 파라미터로 최종 OOF + Test 예측

In [ ]:
bp = study.best_params
BEST_TABNET_PARAMS = dict(
    n_d=bp['n_d'],
    n_a=bp['n_a'],
    n_steps=bp['n_steps'],
    gamma=bp['gamma'],
    n_independent=bp['n_independent'],
    n_shared=bp['n_shared'],
    lambda_sparse=bp['lambda_sparse'],
    optimizer_params=dict(
        lr=bp['lr'],
        weight_decay=bp['weight_decay'],
    ),
    mask_type=bp['mask_type'],
    seed=SEED,
    verbose=0,
)
BEST_BS = bp['batch_size']

oof_tab  = np.zeros(len(X_arr))
test_tab = np.zeros(len(Xt_arr))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_arr), 1):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]
    imputer = SimpleImputer(strategy='median')
    X_tr_i  = imputer.fit_transform(X_tr)
    X_val_i = imputer.transform(X_val)
    Xt_i    = imputer.transform(Xt_arr)
    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr_i)
    X_val_s = scaler.transform(X_val_i)
    Xt_s    = scaler.transform(Xt_i)
    model = TabNetClassifier(**BEST_TABNET_PARAMS)
    model.fit(
        X_tr_s, y_tr,
        eval_set=[(X_val_s, y_val)],
        eval_metric=['auc'],
        max_epochs=200,
        patience=20,
        batch_size=BEST_BS,
        virtual_batch_size=max(64, BEST_BS // 8),
        weights=1,
        num_workers=0,  # macOS 멀티프로세싱 크래시 방지
    )
    oof_tab[val_idx]  = model.predict_proba(X_val_s)[:, 1]
    test_tab         += model.predict_proba(Xt_s)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y_val, oof_tab[val_idx])
    print(f'  Fold {fold}  AUC: {fold_auc:.5f}')

auc_tab = roc_auc_score(y_arr, oof_tab)
print(f'\n[Best TabNet]  OOF AUC: {auc_tab:.5f}')

np.save(CKPT_DIR / 'oof_tab_exp030.npy',  oof_tab)
np.save(CKPT_DIR / 'test_tab_exp030.npy', test_tab)
print('체크포인트 저장 완료')


Early stopping occurred at epoch 114 with best_epoch = 94 and best_val_0_auc = 0.73166
  Fold 1  AUC: 0.73166


## 4. GBDT 3종 (EXP-020 파라미터)

In [ ]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824, num_leaves=266, max_depth=5,
    min_child_samples=166, feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091, lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442, min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED, verbose=False,
    learning_rate=0.018762200842082854, depth=6, l2_leaf_reg=9.186127499619702,
    min_data_in_leaf=19, subsample=0.8171148782837574,
    colsample_bylevel=0.6937054765970498,
)
XGB_PARAMS = dict(
    objective='binary:logistic', eval_metric='auc',
    scale_pos_weight=neg_pos_ratio, random_state=SEED,
    n_estimators=2000, early_stopping_rounds=50,
    learning_rate=0.055231888823083355, max_depth=4, min_child_weight=59,
    subsample=0.7663038394266069, colsample_bytree=0.6581659567709395,
    reg_alpha=8.692166325700154, reg_lambda=0.23929571696793675,
    tree_method='hist', device='cpu',
)

oof_lgb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = model.predict(X_val)
    test_lgb         += model.predict(X_test) / N_FOLDS
auc_lgb = roc_auc_score(y_train, oof_lgb)
print(f'[LightGBM]  OOF AUC: {auc_lgb:.5f}')

In [ ]:
oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = CatBoostClassifier(**CAT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_cat         += model.predict_proba(X_test)[:, 1] / N_FOLDS
auc_cat = roc_auc_score(y_train, oof_cat)
print(f'[CatBoost]  OOF AUC: {auc_cat:.5f}')

In [ ]:
oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_xgb         += model.predict_proba(X_test)[:, 1] / N_FOLDS
auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'[XGBoost]   OOF AUC: {auc_xgb:.5f}')

# 중간 체크포인트 저장
np.save(CKPT_DIR / 'oof_lgb_exp030.npy',  oof_lgb)
np.save(CKPT_DIR / 'oof_cat_exp030.npy',  oof_cat)
np.save(CKPT_DIR / 'oof_xgb_exp030.npy',  oof_xgb)
np.save(CKPT_DIR / 'test_lgb_exp030.npy', test_lgb)
np.save(CKPT_DIR / 'test_cat_exp030.npy', test_cat)
np.save(CKPT_DIR / 'test_xgb_exp030.npy', test_xgb)
print(f'\n  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}  TabNet={auc_tab:.5f}')

## 5. 앙상블 비교

In [ ]:
oofs  = np.stack([oof_lgb,  oof_cat,  oof_xgb,  oof_tab],  axis=1)
tests = np.stack([test_lgb, test_cat, test_xgb, test_tab], axis=1)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb, auc_tab])
n_models = oofs.shape[1]

results = {}

# 1. GBDT only Rank Average (EXP-020 기준선 재확인)
r3_oof  = np.stack([rankdata(oofs[:, i]) for i in range(3)], axis=1)
r3_test = np.stack([rankdata(tests[:, i]) for i in range(3)], axis=1)
results['GBDT-only Rank Avg'] = (
    roc_auc_score(y_train, r3_oof.mean(axis=1)), r3_test.mean(axis=1))

# 2. Simple Average (4모델)
results['Simple Avg (4)'] = (
    roc_auc_score(y_train, oofs.mean(axis=1)), tests.mean(axis=1))

# 3. Rank Average (4모델)
r4_oof  = np.stack([rankdata(oofs[:, i]) for i in range(n_models)], axis=1)
r4_test = np.stack([rankdata(tests[:, i]) for i in range(n_models)], axis=1)
results['Rank Avg (4)'] = (
    roc_auc_score(y_train, r4_oof.mean(axis=1)), r4_test.mean(axis=1))

# 4. Optuna 가중치 (4모델)
def optuna_ensemble_objective(trial):
    w = np.array([
        trial.suggest_float('w_lgb', 0.0, 1.0),
        trial.suggest_float('w_cat', 0.0, 1.0),
        trial.suggest_float('w_xgb', 0.0, 1.0),
        trial.suggest_float('w_tab', 0.0, 1.0),
    ])
    w = w / w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

ens_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
ens_study.optimize(optuna_ensemble_objective, n_trials=300, show_progress_bar=False)
best_w = np.array([
    ens_study.best_params['w_lgb'], ens_study.best_params['w_cat'],
    ens_study.best_params['w_xgb'], ens_study.best_params['w_tab'],
])
best_w = best_w / best_w.sum()
results['Optuna Weights (4)'] = (
    roc_auc_score(y_train, (oofs * best_w).sum(axis=1)),
    (tests * best_w).sum(axis=1)
)

BASELINE = 0.74068
print('=' * 70)
print(f'  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}  TabNet={auc_tab:.5f}')
print(f'  EXP-020 기준선: {BASELINE}')
print('-' * 70)
best_method, best_auc, best_test = '', 0, None
for method, (auc, test_pred) in results.items():
    flag = ' ←' if auc > BASELINE else ''
    print(f'  {method:22s}: {auc:.5f}  ({auc - BASELINE:+.5f} vs EXP-020){flag}')
    if auc > best_auc:
        best_auc, best_method, best_test = auc, method, test_pred
print('=' * 70)
print(f'\n최적: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  '
      f'XGB={best_w[2]:.3f}  TabNet={best_w[3]:.3f}')

## 6. Submission 저장 & 실험 기록

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Rank Average는 rank 값이라 0~1 정규화 필요
if best_test.max() > 1.0:
    best_test = (best_test - best_test.min()) / (best_test.max() - best_test.min())
    print(f'[정규화 적용] {best_method} → 0~1 변환 완료')

sub['probability'] = best_test
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}  ({best_method})')
print(f'probability 범위: [{sub["probability"].min():.4f}, {sub["probability"].max():.4f}]')

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')


if best_method == 'Optuna Weights (4)':
    best_oof_pred = (oofs * best_w).sum(axis=1)
elif best_method == 'Rank Avg (4)':
    best_oof_pred = r4_oof.mean(axis=1)
elif best_method == 'Simple Avg (4)':
    best_oof_pred = oofs.mean(axis=1)
else:
    best_oof_pred = r3_oof.mean(axis=1)
oof_binary = (best_oof_pred >= 0.5).astype(int)

n_trials_done = len(study.trials)
params_str = (
    f'TabNet Optuna {n_trials_done}trials | '
    f'best: n_d={bp["n_d"]} n_a={bp["n_a"]} steps={bp["n_steps"]} '
    f'gamma={bp["gamma"]:.2f} lr={bp["lr"]:.4f} | '
    f'LGB(EXP-019)+CAT(EXP-011)+XGB(EXP-012)+TabNet | '
    f'best={best_method} | '
    f'w=({best_w[0]:.3f},{best_w[1]:.3f},{best_w[2]:.3f},{best_w[3]:.3f})'
)
NOTES    = (f'TabNet Optuna {n_trials_done}trials (timeout={OPTUNA_TIMEOUT//3600:.1f}h) + '
            f'GBDT 3종 앙상블. best={best_method}')
INSIGHTS = (f'EXP-020 대비 {best_auc - BASELINE:+.5f} | '
            f'TabNet OOF={auc_tab:.5f} (Optuna best={study.best_value:.5f}) | '
            f'LGB={auc_lgb:.5f} CAT={auc_cat:.5f} XGB={auc_xgb:.5f} | '
            f'TabNet weight={best_w[3]:.3f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Ensemble(LGB+CAT+XGB+TabNet-Optuna)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'v1', X_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight / TabNet weights=1',
    'N', None, f'notebooks/26_tabnet_optuna_yjcho.ipynb', NOTES, INSIGHTS
)